In [1]:
!pip install mlflow xgboost scikit-learn pandas numpy

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path
import joblib

In [3]:
# load the model files from our project
MODEL_DIR = Path("../ai/model")

existing_model = joblib.load(MODEL_DIR / "xgboost_rul_model.pkl")
scaler = joblib.load(MODEL_DIR / "standard_scaler.pkl")
feature_names = joblib.load(MODEL_DIR / "feature_names.pkl")

print(type(existing_model))
print(len(feature_names), "features")

<class 'xgboost.sklearn.XGBRegressor'>
17 features


c:\Users\Daina Mall\.conda\envs\modeling\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
# generate some dummy data to test with
np.random.seed(42)
n_samples = 1000
n_features = len(feature_names)

X = np.random.randn(n_samples, n_features)
w = np.random.randn(n_features) * 0.5
y = X @ w + np.random.randn(n_samples) * 10 + 125
y = np.clip(y, 0, 300)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("train:", X_train.shape, "test:", X_test.shape)

train: (800, 17) test: (200, 17)


In [5]:
from pathlib import Path

print(Path.cwd())

c:\Users\Daina Mall\Downloads\DEPI-project-RAG-main (1)\DEPI-project-RAG-main\notebooks


In [6]:
from pathlib import Path

db_path = Path.cwd() / "mlflow.db"

print(db_path)
print(db_path.resolve())

c:\Users\Daina Mall\Downloads\DEPI-project-RAG-main (1)\DEPI-project-RAG-main\notebooks\mlflow.db
C:\Users\Daina Mall\Downloads\DEPI-project-RAG-main (1)\DEPI-project-RAG-main\notebooks\mlflow.db


In [7]:
import mlflow
from pathlib import Path

db_path = Path.cwd() / "mlflow.db"

# استخدمي قاعدة البيانات المحلية
mlflow.set_tracking_uri(f"sqlite:///{db_path.resolve().as_posix()}")

print("Tracking URI:", mlflow.get_tracking_uri())
print("DB Path:", db_path.resolve())

Tracking URI: sqlite:///C:/Users/Daina Mall/Downloads/DEPI-project-RAG-main (1)/DEPI-project-RAG-main/notebooks/mlflow.db
DB Path: C:\Users\Daina Mall\Downloads\DEPI-project-RAG-main (1)\DEPI-project-RAG-main\notebooks\mlflow.db


In [8]:
# first training run with mlflow
mlflow.set_experiment("Predictive Maintenance - Demo")

with mlflow.start_run(run_name="run1-default"):
    params = {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "objective": "reg:squarederror"
    }
    mlflow.log_params(params)

    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")

    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

2026/07/14 19:34:48 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/07/14 19:34:48 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

RMSE: 10.59, MAE: 8.54, R2: -0.09


In [9]:
# try different parameters
with mlflow.start_run(run_name="run2-more-trees"):
    params = {"n_estimators": 200, "max_depth": 8, "learning_rate": 0.1, "objective": "reg:squarederror"}
    mlflow.log_params(params)
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

with mlflow.start_run(run_name="run3-low-lr"):
    params = {"n_estimators": 150, "max_depth": 6, "learning_rate": 0.05, "objective": "reg:squarederror"}
    mlflow.log_params(params)
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

with mlflow.start_run(run_name="run4-shallow"):
    params = {"n_estimators": 80, "max_depth": 4, "learning_rate": 0.15, "objective": "reg:squarederror"}
    mlflow.log_params(params)
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.xgboost.log_model(model, "model")
    print(f"RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.2f}")

print("done")

2026/07/14 19:35:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE: 10.42, MAE: 8.21, R2: -0.05


2026/07/14 19:35:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE: 10.30, MAE: 8.31, R2: -0.03


2026/07/14 19:35:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE: 10.43, MAE: 8.50, R2: -0.06
done


In [10]:
# log some prediction stats too
with mlflow.start_run(run_name="batch-prediction"):
    y_pred = model.predict(X_test)

    mlflow.log_metric("avg_rul", float(np.mean(y_pred)))
    mlflow.log_metric("min_rul", float(np.min(y_pred)))
    mlflow.log_metric("max_rul", float(np.max(y_pred)))
    mlflow.log_metric("std_rul", float(np.std(y_pred)))

    critical = int(np.sum(y_pred < 30))
    warning = int(np.sum((y_pred >= 30) & (y_pred < 75)))
    healthy = int(np.sum(y_pred >= 75))

    mlflow.log_metric("engines_critical", critical)
    mlflow.log_metric("engines_warning", warning)
    mlflow.log_metric("engines_healthy", healthy)

    print(f"avg RUL: {np.mean(y_pred):.1f}")
    print(f"critical: {critical}, warning: {warning}, healthy: {healthy}")

avg RUL: 125.4
critical: 0, warning: 0, healthy: 200


In [11]:
# list all runs
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("Predictive Maintenance - Demo")
runs = client.search_runs(experiment.experiment_id)

for run in runs:
    d = run.data
    rmse_val = d.metrics.get('rmse')
    rmse_str = f"{rmse_val:.2f}" if rmse_val is not None else "N/A"
    print(run.info.run_name, "-> RMSE:", rmse_str)

batch-prediction -> RMSE: N/A
run4-shallow -> RMSE: 10.43
run3-low-lr -> RMSE: 10.30
run2-more-trees -> RMSE: 10.42
run1-default -> RMSE: 10.59
batch-prediction -> RMSE: N/A
run4-shallow -> RMSE: 10.43
run3-low-lr -> RMSE: 10.30
run2-more-trees -> RMSE: 10.42
run1-default -> RMSE: 10.59
run1-default -> RMSE: 10.59


In [12]:
# to see the dashboard run this in terminal:
# mlflow ui
# then go to http://localhost:5000